# ⚽ World Cup 2026 Win Probability — Phase 2: Build & Train (v2)

We now **build the football brain** and teach it, using the 74,598 snapshots from Phase 1.

## What this notebook does
```
1. Load the Phase 1 dataset
2. Build a SMALL football-native neural net (11 -> 24 -> 12 -> 3)
3. Warm-start the transferable parts from your NBA model
   -> print a clear BEFORE / AFTER of what carried over
4. Split the data properly (no leakage between train and test)
5. Stage 1: train only the new parts (frozen core)
6. Stage 2: gently fine-tune the whole net
7. Calibrate the probabilities (temperature scaling)
8. Evaluate — overall AND draw-specific
9. Save the model + append results to results/RESULTS.md
```

---
### The big-picture analogy for this whole phase
Imagine you hire someone who is already an **expert basketball coach** and you want to turn them into a **football coach**.

They don't know football's rules — but they already understand *general sport sense*: momentum swings, that a lead late in a game is safer than the same lead early, that a stronger team should usually win but upsets happen. That general wisdom is worth keeping.

So you don't send them to school from scratch. You:
1. Keep their general sport instincts (**warm-start from the NBA weights**).
2. First teach them *only* the football-specific bits while telling them "don't second-guess your general instincts yet" (**Stage 1 — frozen core**).
3. Then let them adjust everything together, gently, now that they've got the basics (**Stage 2 — fine-tune**).

That two-step teaching is the heart of transfer learning, and it's exactly what this notebook does.


## Cell 1 — Setup

In [1]:
import json, time, pickle, warnings
import numpy as np, pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from copy import deepcopy
warnings.filterwarnings('ignore')

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix, classification_report

ROOT      = Path('.')
DATA_PROC = ROOT / 'data' / 'processed'
MODELS    = ROOT / 'models'
RESULTS   = ROOT / 'results'
for d in (MODELS, RESULTS): d.mkdir(parents=True, exist_ok=True)

# CPU is totally fine here — the net is tiny and the data is small
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42); np.random.seed(42)
print('Device:', DEVICE)
print('Torch :', torch.__version__)


Device: cpu
Torch : 2.12.1+cpu


## Cell 2 — Load the Phase 1 data

We load the parquet file and pull out the 11 feature columns (the inputs, called `X`) and the outcome column (the answer, called `y`).

- `X` = the situation (score, time, team strengths, etc.) — what the model *sees*.
- `y` = who actually won (0 away / 1 draw / 2 home) — what the model must *guess*.
- `groups` = which match each snapshot came from — we need this so that all snapshots from one match stay together when we split (more on why in Cell 4).


In [2]:
FEATURE_COLS = ['goal_diff','minute_norm','is_second_half','home_rank_norm',
                'away_rank_norm','rank_diff','is_knockout','lead_changes_norm',
                'is_neutral_venue','score_state','strength_x_time']
TARGET_COL = 'outcome'
N_FEATURES, N_CLASSES = len(FEATURE_COLS), 3

df = pd.read_parquet(DATA_PROC / 'features_v2.parquet')
X = df[FEATURE_COLS].values.astype('float32')
y = df[TARGET_COL].values.astype('int64')
groups = df['match_id'].values

print(f'Loaded {len(df):,} snapshots, {N_FEATURES} features')
print(f'X shape {X.shape}, y shape {y.shape}')
print('Class counts (0 away / 1 draw / 2 home):', np.bincount(y))


Loaded 74,598 snapshots, 11 features
X shape (74598, 11), y shape (74598,)
Class counts (0 away / 1 draw / 2 home): [23838 18809 31951]


## Cell 3 — Build the football brain (the network)

Here's the network, layer by layer. Think of it as a **series of filters that turn 11 raw numbers into 3 probabilities**.

```
11 inputs  ->  24 neurons  ->  12 neurons  ->  3 outputs
   (the       (first hidden   (second hidden   (away / draw
   situation)   layer)          layer)           / home)
```

Why so small (24 then 12)? Because our data is small-ish. Here's the analogy: if you give a student with a *photographic memory* only 20 flashcards, they'll just memorise the 20 cards word-for-word and fail the moment a 21st card appears — they never learned the *idea*, only the cards. A smaller brain literally doesn't have enough room to memorise, so it's *forced* to learn the general pattern instead. That "memorising instead of understanding" is what **overfitting** means, and a small network is our main defence against it.

Some jargon, translated:
- **neuron**: a little unit that takes numbers in, mixes them, and passes a number on.
- **ReLU**: a simple on/off rule — if a neuron's number is negative, it becomes 0; if positive, it passes through. It lets the network learn bends and kinks, not just straight lines.
- **Dropout (0.3)**: during training we randomly switch off 30% of neurons each step. Sounds mad, but it stops the network leaning too hard on any one neuron — like a football team practising with random players sitting out, so no single star becomes a crutch.
- **softmax**: the final step that turns 3 raw scores into 3 probabilities that add up to 100%.


In [3]:
class FootballNet(nn.Module):
    def __init__(self, n_features=11, n_classes=3, h1=24, h2=12, dropout=0.30):
        super().__init__()
        self.fc1  = nn.Linear(n_features, h1)   # input -> hidden 1
        self.fc2  = nn.Linear(h1, h2)           # hidden 1 -> hidden 2
        self.head = nn.Linear(h2, n_classes)    # hidden 2 -> 3 outputs
        self.drop = nn.Dropout(dropout)
        self.act  = nn.ReLU()
    def forward(self, x):
        x = self.drop(self.act(self.fc1(x)))
        x = self.drop(self.act(self.fc2(x)))
        return self.head(x)   # raw logits (softmax applied later)

model = FootballNet(N_FEATURES, N_CLASSES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nTotal learnable numbers in this net: {n_params:,}')


FootballNet(
  (fc1): Linear(in_features=11, out_features=24, bias=True)
  (fc2): Linear(in_features=24, out_features=12, bias=True)
  (head): Linear(in_features=12, out_features=3, bias=True)
  (drop): Dropout(p=0.3, inplace=False)
  (act): ReLU()
)

Total learnable numbers in this net: 627


## Cell 4 — Warm-start from the NBA model (the transfer) + BEFORE/AFTER

Now the interesting part. Your NBA model already learned to read *game situations*. Its very first layer learned to take raw game numbers and produce useful internal signals. We want to donate that head-start to the football net.

**The honest reality of the transfer** (and why the before/after print matters): the two networks are different sizes, so we can't copy everything. We can only sensibly transfer where shapes line up. What we *can* do is copy the overlapping slice of the first layer's weights — the part that reads the shared "game state" features (score gap, time, team strength). The rest stays freshly initialised for football.

The analogy: our basketball coach's notebook is written in a slightly different format than the football notebook. We can't photocopy the whole thing — but we *can* copy over the pages about universal ideas (momentum, time pressure, favourite vs underdog) into the matching sections. The football-only pages stay blank for now, to be filled in during training.

The cell below prints, for the first layer, the actual numbers **before** the transfer (fresh random) and **after** (NBA-donated), so you can see with your own eyes that real values were carried over, not just trust me.


In [4]:
# ---- Locate the NBA weights file ----
nba_candidates = [
    MODELS / 'nba_base.pth',
    Path("C:\\Users\\rhkha\\Documents\\Documents\\Schoolwork\\Projects\\NBA LIVE PREDICTIONS\\nba_win_prob\\model\\win_prob_net.pth"),
    Path('nba_win_prob/model/win_prob_net.pth'),
]
nba_path = next((p for p in nba_candidates if p.exists()), None)

if nba_path is None:
    print('NBA weights not found in any of:')
    for p in nba_candidates: print('   ', p)
    print('\n>> Copy your NBA win_prob_net.pth to models/nba_base.pth and re-run this cell.')
    print('>> (Training still works without it — it just starts from scratch instead of warm.)')
    TRANSFER_DONE = False
else:
    print('Found NBA weights at:', nba_path)
    nba_ckpt  = torch.load(nba_path, map_location='cpu', weights_only=False)
    nba_state = nba_ckpt['model_state_dict'] if 'model_state_dict' in nba_ckpt else nba_ckpt

    # Find the NBA first-layer weight matrix (shape [hidden, nba_features])
    nba_first_w = None
    for k, v in nba_state.items():
        if v.ndim == 2 and v.shape[1] >= 7:   # first Linear reading the features
            nba_first_w = v; nba_first_key = k; break

    # ---- BEFORE ----
    before = model.fc1.weight.data.clone()
    print('\n' + '='*60)
    print('BEFORE transfer — football fc1 weights (fresh random)')
    print('='*60)
    print('fc1 shape:', tuple(model.fc1.weight.shape), '(24 neurons x 11 features)')
    print('First neuron, first 5 feature-weights:')
    print('  ', np.round(before[0, :5].numpy(), 4))

    if nba_first_w is not None:
        # Copy the overlapping slice: min(rows), min(cols)
        r = min(model.fc1.weight.shape[0], nba_first_w.shape[0])
        c = min(model.fc1.weight.shape[1], nba_first_w.shape[1])
        with torch.no_grad():
            model.fc1.weight.data[:r, :c] = nba_first_w[:r, :c].float()
        print(f'\nDonated NBA slice: {r} neurons x {c} features '
              f'(from NBA layer "{nba_first_key}" shape {tuple(nba_first_w.shape)})')

    # ---- AFTER ----
    after = model.fc1.weight.data.clone()
    print('\n' + '='*60)
    print('AFTER transfer — same weights now NBA-warmed')
    print('='*60)
    print('First neuron, first 5 feature-weights:')
    print('  ', np.round(after[0, :5].numpy(), 4))
    changed = int((before != after).sum().item())
    total   = before.numel()
    print(f'\nWeights changed by transfer: {changed:,} / {total:,} '
          f'({changed/total:.0%} of fc1)')
    print('The rest stay fresh — those are the football-only pages, filled in during training.')
    TRANSFER_DONE = True


Found NBA weights at: C:\Users\rhkha\Documents\Documents\Schoolwork\Projects\NBA LIVE PREDICTIONS\nba_win_prob\model\win_prob_net.pth

BEFORE transfer — football fc1 weights (fresh random)
fc1 shape: (24, 11) (24 neurons x 11 features)
First neuron, first 5 feature-weights:
   [ 0.2305  0.2503 -0.0706  0.277  -0.0661]

Donated NBA slice: 24 neurons x 11 features (from NBA layer "net.0.weight" shape (128, 12))

AFTER transfer — same weights now NBA-warmed
First neuron, first 5 feature-weights:
   [ 0.4665  0.2818  0.4175  0.1978 -0.3009]

Weights changed by transfer: 264 / 264 (100% of fc1)
The rest stay fresh — those are the football-only pages, filled in during training.


## Cell 5 — Split the data safely (no cheating)

We split into three piles:
- **train** (~60%) — the model studies these.
- **validation** (~20%) — we peek at these during training to tune things, but the model never studies them directly.
- **test** (~20%) — locked in a vault, only opened at the very end for the honest grade.

**The critical rule: no single match may have snapshots in two different piles.** Why? Because all ~20 snapshots from one match share the same final outcome. If snapshot #5 of a match were in "train" and snapshot #6 in "test", the model would have basically seen the answer to the test question during study — that's cheating, and it makes the model look better than it really is. This is called **data leakage**. Keeping whole matches together (`GroupShuffleSplit` on `match_id`) slams that door shut.


In [5]:
# Split by MATCH, not by row, so no match spans two piles
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
trainval_idx, test_idx = next(gss1.split(X, y, groups))

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_rel, va_rel = next(gss2.split(X[trainval_idx], y[trainval_idx], groups[trainval_idx]))
train_idx = trainval_idx[tr_rel]
val_idx   = trainval_idx[va_rel]

# Proof there's no overlap of matches between piles
tr_m, va_m, te_m = set(groups[train_idx]), set(groups[val_idx]), set(groups[test_idx])
assert not (tr_m & va_m) and not (tr_m & te_m) and not (va_m & te_m)
print('No match appears in two piles — leakage check passed.')

# Scale features: turn every column into 'how many std-devs from average'
# so no single feature dominates just because its raw numbers are bigger.
scaler = StandardScaler().fit(X[train_idx])
Xtr = scaler.transform(X[train_idx]).astype('float32')
Xva = scaler.transform(X[val_idx]).astype('float32')
Xte = scaler.transform(X[test_idx]).astype('float32')
ytr, yva, yte = y[train_idx], y[val_idx], y[test_idx]

print(f'train {len(train_idx):,}  |  val {len(val_idx):,}  |  test {len(test_idx):,} snapshots')
print(f'train {len(tr_m)}  |  val {len(va_m)}  |  test {len(te_m)} matches')

def loader(Xa, ya, bs=256, shuffle=False):
    ds = TensorDataset(torch.tensor(Xa), torch.tensor(ya))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle)
train_loader = loader(Xtr, ytr, shuffle=True)
val_loader   = loader(Xva, yva)


No match appears in two piles — leakage check passed.
train 44,785  |  val 14,901  |  test 14,912 snapshots
train 2178  |  val 726  |  test 726 matches


## Cell 6 — Handle the draw problem with class weights

Draws are the minority (only ~25% of matches) AND the hardest to predict. Left alone, the model takes the lazy route: "I'll rarely predict a draw, and I'll still be right most of the time." That's like a doctor who never diagnoses a rare disease — they're right often, but useless exactly when it matters.

We fight this with **class weights**: we tell the loss function "getting a draw wrong should hurt more than getting a home/away wrong." The weight for each class is the inverse of how common it is, so the rare draw class gets a bigger penalty. Worked example, every step:
- Suppose in the training data: away=32%, draw=25%, home=43% (as fractions: 0.32, 0.25, 0.43).
- Inverse of each: 1/0.32 = 3.13, 1/0.25 = 4.00, 1/0.43 = 2.33.
- The draw class gets the biggest weight (4.00) because it's rarest — so the model is punished hardest for ignoring draws.


In [6]:
counts = np.bincount(ytr, minlength=3).astype('float64')
freqs  = counts / counts.sum()
weights = 1.0 / freqs
weights = weights / weights.sum() * 3.0   # normalise so they average ~1
class_weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
print('Class      away    draw    home')
print('Frequency ', ' '.join(f'{f:6.1%}' for f in freqs))
print('Weight    ', ' '.join(f'{w:6.2f}' for w in weights))
print('\n-> draw has the largest weight, so ignoring draws is punished most.')


Class      away    draw    home
Frequency   31.7%  26.4%  41.9%
Weight       1.02   1.22   0.77

-> draw has the largest weight, so ignoring draws is punished most.


## Cell 7 — Stage 1: teach only the new football parts (core frozen)

Now the first teaching stage. We **freeze** the warm-started first layer — "freeze" means we tell it *don't change during this stage*. Only the new football-specific layers (fc2 and the output head) are allowed to learn.

Why? The first layer arrived with useful NBA instincts. If we let *everything* change immediately, the fresh random football layers would send garbage feedback that could scramble those good instincts before they're useful. So Stage 1 says: "core, hold still and keep your instincts; new layers, you do the learning for now."

The analogy: our basketball coach keeps their general instincts fixed while they *first* memorise football's basic rules. Only once they know the rules do we let them blend rules with instinct (that's Stage 2).


In [7]:
def run_epoch(net, loader, optimizer=None):
    train_mode = optimizer is not None
    net.train() if train_mode else net.eval()
    crit = nn.CrossEntropyLoss(weight=class_weights)
    total_loss, all_logits, all_y = 0.0, [], []
    with torch.set_grad_enabled(train_mode):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = net(xb)
            loss = crit(logits, yb)
            if train_mode:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(xb)
            all_logits.append(logits.detach().cpu()); all_y.append(yb.cpu())
    logits = torch.cat(all_logits); yv = torch.cat(all_y)
    acc = (logits.argmax(1) == yv).float().mean().item()
    return total_loss / len(loader.dataset), acc

# Freeze the warm-started first layer
model.fc1.weight.requires_grad = False
model.fc1.bias.requires_grad   = False
trainable = [p for p in model.parameters() if p.requires_grad]
print('Stage 1: fc1 FROZEN. Training fc2 + head only.')
print('Trainable tensors:', len(trainable))

opt1 = torch.optim.Adam(trainable, lr=3e-3)
for epoch in range(1, 13):
    tr_loss, tr_acc = run_epoch(model, train_loader, opt1)
    va_loss, va_acc = run_epoch(model, val_loader)
    if epoch % 3 == 0 or epoch == 1:
        print(f'  epoch {epoch:2d}  train_acc {tr_acc:.3f}  val_acc {va_acc:.3f}')
print('Stage 1 done.')


Stage 1: fc1 FROZEN. Training fc2 + head only.
Trainable tensors: 4
  epoch  1  train_acc 0.474  val_acc 0.481
  epoch  3  train_acc 0.486  val_acc 0.469
  epoch  6  train_acc 0.486  val_acc 0.467
  epoch  9  train_acc 0.486  val_acc 0.468
  epoch 12  train_acc 0.488  val_acc 0.472
Stage 1 done.


## Cell 8 — Stage 2: gently fine-tune everything together

Now we **unfreeze** the first layer so the whole network can adjust as one. But we do it at a much *smaller* learning rate.

"Learning rate" = how big a step the model takes each time it corrects itself. Analogy: tuning a guitar. Big turns of the peg (big learning rate) get you in the rough area fast but overshoot the exact note. Tiny turns (small learning rate) dial in the precise pitch without snapping the string. Stage 1 used big turns to get close; Stage 2 uses tiny turns (learning rate 10x smaller) so the network refines without wrecking the NBA instincts we carefully preserved.


In [8]:
for p in model.parameters():
    p.requires_grad = True
print('Stage 2: all layers unfrozen. Fine-tuning gently (lr 3e-4).')

opt2 = torch.optim.Adam(model.parameters(), lr=3e-4)
best_val, best_state, patience, bad = 1e9, None, 6, 0
for epoch in range(1, 41):
    tr_loss, tr_acc = run_epoch(model, train_loader, opt2)
    va_loss, va_acc = run_epoch(model, val_loader)
    if va_loss < best_val - 1e-4:
        best_val, best_state, bad = va_loss, deepcopy(model.state_dict()), 0
        star = ' *'
    else:
        bad += 1; star = ''
    if epoch % 4 == 0 or epoch == 1 or star:
        print(f'  epoch {epoch:2d}  train_acc {tr_acc:.3f}  val_acc {va_acc:.3f}  val_loss {va_loss:.4f}{star}')
    if bad >= patience:
        print(f'  early stop at epoch {epoch} (val stopped improving)'); break

model.load_state_dict(best_state)
print('Stage 2 done — restored best-validation weights.')


Stage 2: all layers unfrozen. Fine-tuning gently (lr 3e-4).
  epoch  1  train_acc 0.490  val_acc 0.472  val_loss 0.9363 *
  epoch  2  train_acc 0.489  val_acc 0.472  val_loss 0.9353 *
  epoch  3  train_acc 0.490  val_acc 0.471  val_loss 0.9344 *
  epoch  4  train_acc 0.490  val_acc 0.471  val_loss 0.9338 *
  epoch  5  train_acc 0.490  val_acc 0.472  val_loss 0.9329 *
  epoch  6  train_acc 0.490  val_acc 0.472  val_loss 0.9323 *
  epoch  8  train_acc 0.489  val_acc 0.471  val_loss 0.9323
  epoch  9  train_acc 0.490  val_acc 0.471  val_loss 0.9318 *
  epoch 10  train_acc 0.490  val_acc 0.471  val_loss 0.9312 *
  epoch 11  train_acc 0.491  val_acc 0.471  val_loss 0.9306 *
  epoch 12  train_acc 0.490  val_acc 0.472  val_loss 0.9301 *
  epoch 13  train_acc 0.491  val_acc 0.472  val_loss 0.9296 *
  epoch 14  train_acc 0.491  val_acc 0.472  val_loss 0.9289 *
  epoch 15  train_acc 0.490  val_acc 0.471  val_loss 0.9286 *
  epoch 16  train_acc 0.490  val_acc 0.472  val_loss 0.9289
  epoch 17  tr

## Cell 9 — Calibrate the probabilities (temperature scaling)

A trained network is often *overconfident* — it says "80% home win" when really it should say "68%." Calibration fixes the confidence without changing which outcome it picks.

We find one number, **T** (temperature), and divide the raw scores by it before softmax:
- T = 1 -> no change.
- T > 1 -> softens overconfident predictions (pulls them toward the middle).
- We search for the T that makes the validation predictions *most honest* (lowest log-loss).

Analogy: a weather app that says "100% rain" and then it's dry is badly calibrated. Temperature scaling is the reality-check dial that turns those cocky 100%s into believable 80%s — so when the model says 70%, it really means 7-in-10.


In [9]:
model.eval()
with torch.no_grad():
    val_logits = model(torch.tensor(Xva).to(DEVICE)).cpu().numpy()

def logloss_at_T(T):
    z = val_logits / T
    z = z - z.max(1, keepdims=True)
    p = np.exp(z); p = p / p.sum(1, keepdims=True)
    p = np.clip(p, 1e-7, 1-1e-7)
    return log_loss(yva, p, labels=[0,1,2])

Ts = np.linspace(0.5, 3.0, 60)
losses = [logloss_at_T(T) for T in Ts]
T_best = float(Ts[int(np.argmin(losses))])
print(f'Best temperature T = {T_best:.3f}')
print(f'  log-loss before (T=1): {logloss_at_T(1.0):.4f}')
print(f'  log-loss after  (T={T_best:.2f}): {logloss_at_T(T_best):.4f}  (lower = more honest)')


Best temperature T = 0.881
  log-loss before (T=1): 0.9177
  log-loss after  (T=0.88): 0.9167  (lower = more honest)


## Cell 10 — The honest grade: evaluate on the locked test vault

Now we open the test pile — matches the model has never touched — and grade it. We look at two things:
1. **Overall accuracy**: of all snapshots, what fraction did it call right?
2. **The draw problem specifically**: how well does it handle the hard class? The confusion matrix shows this — each row is the true outcome, each column what the model guessed. A perfect model has all its numbers on the diagonal (guess = truth).

Remember the baseline: always guessing "home win" would score ~43%. And the research says even great football models only reach the mid-50s%, with draws dragging things down. So don't expect NBA-level 76% — expect 'better than 43%, respectable for football, weak on draws.' That's the honest bar.


In [10]:
def probs_at_T(logits, T):
    z = logits / T; z = z - z.max(1, keepdims=True)
    p = np.exp(z); return p / p.sum(1, keepdims=True)

with torch.no_grad():
    test_logits = model(torch.tensor(Xte).to(DEVICE)).cpu().numpy()
test_p    = probs_at_T(test_logits, T_best)
test_pred = test_p.argmax(1)

overall_acc = accuracy_score(yte, test_pred)
baseline_acc = max(np.bincount(yte)) / len(yte)
names = ['away','draw','home']

print(f'Overall test accuracy : {overall_acc:.3f}')
print(f'Baseline (always home): {baseline_acc:.3f}')
print(f'Test log-loss         : {log_loss(yte, test_p, labels=[0,1,2]):.4f}\n')

print('Confusion matrix (rows = truth, cols = guess):')
cm = confusion_matrix(yte, test_pred, labels=[0,1,2])
print('           guess_away  guess_draw  guess_home')
for i, n in enumerate(names):
    print(f'  true_{n}  ' + ''.join(f'{cm[i][j]:11d}' for j in range(3)))

print('\nPer-class report:')
print(classification_report(yte, test_pred, target_names=names, digits=3, zero_division=0))

# Draw-specific recall = of all real draws, how many did we catch?
draw_recall = cm[1,1] / cm[1].sum() if cm[1].sum() else 0
print(f'Draw recall (caught {cm[1,1]} of {cm[1].sum()} real draws): {draw_recall:.3f}')


Overall test accuracy : 0.470
Baseline (always home): 0.434
Test log-loss         : 0.9251

Confusion matrix (rows = truth, cols = guess):
           guess_away  guess_draw  guess_home
  true_away         1775       2960        209
  true_draw          278       2767        456
  true_home          211       3793       2463

Per-class report:
              precision    recall  f1-score   support

        away      0.784     0.359     0.493      4944
        draw      0.291     0.790     0.425      3501
        home      0.787     0.381     0.513      6467

    accuracy                          0.470     14912
   macro avg      0.621     0.510     0.477     14912
weighted avg      0.670     0.470     0.486     14912

Draw recall (caught 2767 of 3501 real draws): 0.790


## Cell 11 — Save the model + log to the permanent record

In [11]:
# Save model, scaler, temperature together
torch.save({
    'model_state': model.state_dict(),
    'feature_cols': FEATURE_COLS,
    'arch': {'n_features': N_FEATURES, 'n_classes': N_CLASSES, 'h1':24, 'h2':12},
    'temperature': T_best,
    'warm_started': bool(TRANSFER_DONE),
}, MODELS / 'football_v1.pth')
with open(MODELS / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('Saved models/football_v1.pth and models/scaler.pkl')

# Append to the permanent track record
stamp = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')
entry = f'''
## Phase 2 — build & train ({stamp})
- Architecture: 11 -> 24 -> 12 -> 3  ({sum(p.numel() for p in model.parameters()):,} params)
- Warm-started from NBA weights: {TRANSFER_DONE}
- Test accuracy: {overall_acc:.3f}  (baseline always-home: {baseline_acc:.3f})
- Test log-loss: {log_loss(yte, test_p, labels=[0,1,2]):.4f}
- Temperature T: {T_best:.3f}
- Draw recall: {draw_recall:.3f}  (the hard class)
- Saved: models/football_v1.pth
'''
rp = RESULTS / 'RESULTS.md'
if not rp.exists():
    rp.write_text('# World Cup 2026 Win Probability — Results Log\n')
with rp.open('a') as f: f.write(entry)

mc = RESULTS / 'metrics.csv'
row = pd.DataFrame([{'timestamp':stamp,'phase':'phase2_train',
    'test_acc':round(overall_acc,4),'baseline_acc':round(baseline_acc,4),
    'test_logloss':round(log_loss(yte,test_p,labels=[0,1,2]),4),
    'draw_recall':round(draw_recall,4),'temperature':round(T_best,3)}])
row.to_csv(mc, mode='a', header=not mc.exists(), index=False)

print(entry)
print('PHASE 2 COMPLETE  ->  next: phase3_wc_validation.ipynb')


Saved models/football_v1.pth and models/scaler.pkl

## Phase 2 — build & train (2026-07-04 09:37 UTC)
- Architecture: 11 -> 24 -> 12 -> 3  (627 params)
- Warm-started from NBA weights: True
- Test accuracy: 0.470  (baseline always-home: 0.434)
- Test log-loss: 0.9251
- Temperature T: 0.881
- Draw recall: 0.790  (the hard class)
- Saved: models/football_v1.pth

PHASE 2 COMPLETE  ->  next: phase3_wc_validation.ipynb
